# Kalman Filter: From Scratch Implementation to Advanced Applications

## Learning Objectives

By completing this notebook, you will:
1. Understand the complete Kalman filter algorithm (predict and update steps)
2. Implement Kalman filter from scratch for tracking and estimation
3. Distinguish between filtering (online) and smoothing (offline) 
4. Apply Kalman filtering to commodity price tracking with noisy data
5. Diagnose filter performance using innovations analysis

## Prerequisites
- Linear algebra (matrix operations)
- State space model basics (Module 3.1)
- Bayesian inference fundamentals

## Estimated Time: 90 minutes

---

## 1. Setup and Imports

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Optional: statsmodels for validation
from statsmodels.tsa.statespace.structural import UnobservedComponents

# Configuration
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("Environment ready!")

## 2. Kalman Filter Theory

### State Space Representation

**State equation:**
$$\mathbf{x}_t = \mathbf{F}_t \mathbf{x}_{t-1} + \mathbf{G}_t \mathbf{w}_t, \quad \mathbf{w}_t \sim \mathcal{N}(0, \mathbf{Q}_t)$$

**Observation equation:**
$$\mathbf{y}_t = \mathbf{H}_t \mathbf{x}_t + \mathbf{v}_t, \quad \mathbf{v}_t \sim \mathcal{N}(0, \mathbf{R}_t)$$

### Kalman Filter Algorithm

At each time $t$:

**Prediction Step:**
$$\begin{align}
\mathbf{x}_{t|t-1} &= \mathbf{F}_t \mathbf{x}_{t-1|t-1} && \text{(State prediction)} \\
\mathbf{P}_{t|t-1} &= \mathbf{F}_t \mathbf{P}_{t-1|t-1} \mathbf{F}_t^T + \mathbf{G}_t \mathbf{Q}_t \mathbf{G}_t^T && \text{(Covariance prediction)}
\end{align}$$

**Update Step:**
$$\begin{align}
\mathbf{v}_t &= \mathbf{y}_t - \mathbf{H}_t \mathbf{x}_{t|t-1} && \text{(Innovation)} \\
\mathbf{S}_t &= \mathbf{H}_t \mathbf{P}_{t|t-1} \mathbf{H}_t^T + \mathbf{R}_t && \text{(Innovation covariance)} \\
\mathbf{K}_t &= \mathbf{P}_{t|t-1} \mathbf{H}_t^T \mathbf{S}_t^{-1} && \text{(Kalman gain)} \\
\mathbf{x}_{t|t} &= \mathbf{x}_{t|t-1} + \mathbf{K}_t \mathbf{v}_t && \text{(State update)} \\
\mathbf{P}_{t|t} &= (\mathbf{I} - \mathbf{K}_t \mathbf{H}_t) \mathbf{P}_{t|t-1} && \text{(Covariance update)}
\end{align}$$

### Key Insight

Kalman gain $\mathbf{K}_t$ balances:
- **Low $\mathbf{K}_t$**: Trust prediction more than measurement
- **High $\mathbf{K}_t$**: Trust measurement more than prediction

## 3. Scalar Kalman Filter (1D State, 1D Observation)

Start with simplest case: tracking a constant with noisy measurements.

In [ ]:
class ScalarKalmanFilter:
    """
    Simple Kalman filter for 1D state, 1D observation.
    
    State model: x_t = F * x_{t-1} + w_t,  w_t ~ N(0, Q)
    Obs model:   y_t = H * x_t + v_t,      v_t ~ N(0, R)
    """
    
    def __init__(self, F, H, Q, R, x0, P0):
        """
        Parameters
        ----------
        F : float
            State transition (e.g., 1.0 for random walk)
        H : float
            Observation matrix (e.g., 1.0 for direct observation)
        Q : float
            Process noise variance
        R : float
            Measurement noise variance
        x0 : float
            Initial state estimate
        P0 : float
            Initial state variance
        """
        self.F = F
        self.H = H
        self.Q = Q
        self.R = R
        
        self.x = x0  # Current state estimate
        self.P = P0  # Current state variance
        
        # Storage
        self.history = {
            'x_pred': [],
            'P_pred': [],
            'x_filt': [],
            'P_filt': [],
            'K': [],
            'innovation': []
        }
    
    def predict(self):
        """Prediction step."""
        self.x_pred = self.F * self.x
        self.P_pred = self.F * self.P * self.F + self.Q
        
        self.history['x_pred'].append(self.x_pred)
        self.history['P_pred'].append(self.P_pred)
    
    def update(self, y):
        """Update step with new measurement."""
        # Innovation
        innovation = y - self.H * self.x_pred
        S = self.H * self.P_pred * self.H + self.R
        
        # Kalman gain
        K = self.P_pred * self.H / S
        
        # Update estimates
        self.x = self.x_pred + K * innovation
        self.P = (1 - K * self.H) * self.P_pred
        
        # Store
        self.history['x_filt'].append(self.x)
        self.history['P_filt'].append(self.P)
        self.history['K'].append(K)
        self.history['innovation'].append(innovation)
    
    def step(self, y):
        """Complete filter step (predict + update)."""
        self.predict()
        self.update(y)
        return self.x, self.P

print("ScalarKalmanFilter class defined")

In [ ]:
# Test on constant tracking problem
# True value is constant at 100, but measurements are noisy

np.random.seed(42)
T = 50
true_value = 100.0
measurement_noise_std = 10.0

# Generate noisy measurements
measurements = true_value + np.random.normal(0, measurement_noise_std, T)

# Initialize filter
kf = ScalarKalmanFilter(
    F=1.0,           # State is constant
    H=1.0,           # Direct observation
    Q=0.1,           # Very small process noise (nearly constant)
    R=measurement_noise_std**2,  # Known measurement variance
    x0=measurements[0],  # Initialize with first measurement
    P0=100.0         # High initial uncertainty
)

# Run filter
for y in measurements:
    kf.step(y)

print(f"Filter ran for {T} steps")
print(f"Final estimate: {kf.x:.2f} (true: {true_value:.2f})")
print(f"Final variance: {kf.P:.2f}")

In [ ]:
# Visualize filter performance
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

time = np.arange(T)
x_filt = np.array(kf.history['x_filt'])
P_filt = np.array(kf.history['P_filt'])
std_filt = np.sqrt(P_filt)

# Panel 1: Estimates
axes[0].plot(time, measurements, 'o', alpha=0.5, markersize=5, label='Noisy Measurements')
axes[0].plot(time, x_filt, 'r-', linewidth=2, label='Kalman Filter Estimate')
axes[0].fill_between(time, x_filt - 2*std_filt, x_filt + 2*std_filt,
                     alpha=0.2, color='red', label='95% Confidence')
axes[0].axhline(true_value, color='blue', linestyle='--', linewidth=2, label='True Value')
axes[0].set_ylabel('Value', fontsize=12)
axes[0].set_title('Kalman Filter: Tracking Constant from Noisy Data', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Panel 2: Kalman gain
K_history = np.array(kf.history['K'])
axes[1].plot(time, K_history, 'g-', linewidth=2)
axes[1].set_ylabel('Kalman Gain', fontsize=12)
axes[1].set_title('Kalman Gain Evolution (decreases as uncertainty reduces)', fontsize=13)
axes[1].grid(True, alpha=0.3)

# Panel 3: Innovations
innovations = np.array(kf.history['innovation'])
axes[2].plot(time, innovations, 'o-', linewidth=1, markersize=4, alpha=0.7)
axes[2].axhline(0, color='black', linestyle='--', linewidth=1)
axes[2].set_xlabel('Time', fontsize=12)
axes[2].set_ylabel('Innovation', fontsize=12)
axes[2].set_title('Innovations (should be white noise)', fontsize=13)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key Observations:**
1. Filter smooths noisy measurements
2. Kalman gain decreases over time (learning)
3. Innovations appear random (good model fit)

## 4. Multivariate Kalman Filter

Extend to vector states (e.g., position + velocity tracking).

In [ ]:
class KalmanFilter:
    """
    General multivariate Kalman filter.
    """
    
    def __init__(self, F, H, Q, R, x0, P0):
        """
        Parameters
        ----------
        F : ndarray (n_state, n_state)
            State transition matrix
        H : ndarray (n_obs, n_state)
            Observation matrix
        Q : ndarray (n_state, n_state)
            Process noise covariance
        R : ndarray (n_obs, n_obs)
            Measurement noise covariance
        x0 : ndarray (n_state,)
            Initial state
        P0 : ndarray (n_state, n_state)
            Initial covariance
        """
        self.F = F
        self.H = H
        self.Q = Q
        self.R = R
        
        self.x = x0
        self.P = P0
        
        self.n_state = len(x0)
        self.n_obs = H.shape[0]
        
        # Storage
        self.history = {
            'x_filt': [],
            'P_filt': [],
            'K': []
        }
    
    def predict(self):
        """Prediction step."""
        self.x_pred = self.F @ self.x
        self.P_pred = self.F @ self.P @ self.F.T + self.Q
    
    def update(self, y):
        """Update step."""
        # Innovation
        innovation = y - self.H @ self.x_pred
        S = self.H @ self.P_pred @ self.H.T + self.R
        
        # Kalman gain
        K = self.P_pred @ self.H.T @ np.linalg.inv(S)
        
        # Update
        self.x = self.x_pred + K @ innovation
        self.P = (np.eye(self.n_state) - K @ self.H) @ self.P_pred
        
        # Store
        self.history['x_filt'].append(self.x.copy())
        self.history['P_filt'].append(self.P.copy())
        self.history['K'].append(K.copy())
    
    def step(self, y):
        """Complete step."""
        self.predict()
        self.update(y)
        return self.x, self.P

print("KalmanFilter class defined")

In [ ]:
# Example: Track position + velocity from noisy position measurements
# State: [position, velocity]
# Observation: [position]

np.random.seed(42)
T = 100
dt = 1.0  # Time step

# True dynamics
true_position = np.zeros(T)
true_velocity = np.zeros(T)
true_velocity[0] = 2.0  # Initial velocity

for t in range(1, T):
    true_velocity[t] = true_velocity[t-1] + np.random.normal(0, 0.1)
    true_position[t] = true_position[t-1] + true_velocity[t] * dt

# Noisy observations (position only)
obs_noise_std = 5.0
observations = true_position + np.random.normal(0, obs_noise_std, T)

# Define Kalman filter
# State transition: position_{t} = position_{t-1} + velocity_{t-1} * dt
#                  velocity_{t} = velocity_{t-1}
F = np.array([
    [1, dt],
    [0, 1]
])

# Observation: measure position only
H = np.array([[1, 0]])

# Process noise (small velocity changes)
Q = np.array([
    [0.01, 0],
    [0, 0.01]
])

# Measurement noise
R = np.array([[obs_noise_std**2]])

# Initialize
x0 = np.array([observations[0], 0])  # Unknown initial velocity
P0 = np.array([
    [100, 0],
    [0, 100]
])

kf_mv = KalmanFilter(F, H, Q, R, x0, P0)

# Run filter
for y in observations:
    kf_mv.step(np.array([y]))

print(f"Multivariate filter complete")
print(f"Final position estimate: {kf_mv.x[0]:.2f} (true: {true_position[-1]:.2f})")
print(f"Final velocity estimate: {kf_mv.x[1]:.2f} (true: {true_velocity[-1]:.2f})")

In [ ]:
# Visualize tracking results
x_history = np.array(kf_mv.history['x_filt'])
P_history = np.array(kf_mv.history['P_filt'])

position_est = x_history[:, 0]
velocity_est = x_history[:, 1]
position_std = np.sqrt(P_history[:, 0, 0])
velocity_std = np.sqrt(P_history[:, 1, 1])

time = np.arange(T)

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Position
axes[0].plot(time, observations, 'o', alpha=0.3, markersize=3, label='Noisy Observations')
axes[0].plot(time, position_est, 'r-', linewidth=2, label='Filtered Position')
axes[0].fill_between(time, position_est - 2*position_std, position_est + 2*position_std,
                     alpha=0.2, color='red')
axes[0].plot(time, true_position, 'b--', linewidth=2, label='True Position')
axes[0].set_ylabel('Position', fontsize=12)
axes[0].set_title('Kalman Filter: Position + Velocity Tracking', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Velocity
axes[1].plot(time, velocity_est, 'r-', linewidth=2, label='Filtered Velocity')
axes[1].fill_between(time, velocity_est - 2*velocity_std, velocity_est + 2*velocity_std,
                     alpha=0.2, color='red')
axes[1].plot(time, true_velocity, 'b--', linewidth=2, label='True Velocity')
axes[1].set_xlabel('Time', fontsize=12)
axes[1].set_ylabel('Velocity', fontsize=12)
axes[1].set_title('Velocity Estimation (unobserved, inferred from position changes)', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key Insight:** Filter infers velocity from position changes (even though velocity is never directly observed)

## 5. Filtering vs Smoothing

**Filtering**: Uses data up to time $t$ → $p(x_t | y_{1:t})$
- **Online** (real-time)
- For trading, nowcasting

**Smoothing**: Uses all data → $p(x_t | y_{1:T})$
- **Offline** (batch)
- Better estimates (uses future info)
- For analysis, parameter estimation

**Rauch-Tung-Striebel (RTS) smoother:**
$$\begin{align}
\mathbf{x}_{t|T} &= \mathbf{x}_{t|t} + \mathbf{C}_t (\mathbf{x}_{t+1|T} - \mathbf{x}_{t+1|t}) \\
\mathbf{C}_t &= \mathbf{P}_{t|t} \mathbf{F}^T \mathbf{P}_{t+1|t}^{-1}
\end{align}$$

Run backward after forward filter.

In [ ]:
def rts_smoother(x_filt, P_filt, F, Q):
    """
    Rauch-Tung-Striebel smoother.
    
    Parameters
    ----------
    x_filt : list of ndarray
        Filtered states
    P_filt : list of ndarray
        Filtered covariances
    F : ndarray
        State transition matrix
    Q : ndarray
        Process noise covariance
    
    Returns
    -------
    x_smooth : list of ndarray
        Smoothed states
    P_smooth : list of ndarray
        Smoothed covariances
    """
    T = len(x_filt)
    n_state = len(x_filt[0])
    
    x_smooth = [None] * T
    P_smooth = [None] * T
    
    # Initialize with final filtered values
    x_smooth[-1] = x_filt[-1].copy()
    P_smooth[-1] = P_filt[-1].copy()
    
    # Backward pass
    for t in range(T-2, -1, -1):
        # Predicted next state
        x_pred = F @ x_filt[t]
        P_pred = F @ P_filt[t] @ F.T + Q
        
        # Smoother gain
        C = P_filt[t] @ F.T @ np.linalg.inv(P_pred)
        
        # Smooth
        x_smooth[t] = x_filt[t] + C @ (x_smooth[t+1] - x_pred)
        P_smooth[t] = P_filt[t] + C @ (P_smooth[t+1] - P_pred) @ C.T
    
    return x_smooth, P_smooth

# Apply smoother to previous example
x_smooth, P_smooth = rts_smoother(
    kf_mv.history['x_filt'],
    kf_mv.history['P_filt'],
    F, Q
)

x_smooth = np.array(x_smooth)
P_smooth = np.array(P_smooth)

print("Smoothing complete")

In [ ]:
# Compare filtering vs smoothing
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

position_smooth = x_smooth[:, 0]
velocity_smooth = x_smooth[:, 1]
position_std_smooth = np.sqrt(P_smooth[:, 0, 0])

# Position comparison
axes[0].plot(time, observations, 'o', alpha=0.2, markersize=3, label='Observations')
axes[0].plot(time, position_est, 'r-', linewidth=2, alpha=0.7, label='Filtered')
axes[0].plot(time, position_smooth, 'g-', linewidth=2, label='Smoothed')
axes[0].plot(time, true_position, 'b--', linewidth=2, label='True')
axes[0].set_ylabel('Position', fontsize=12)
axes[0].set_title('Filtering vs Smoothing: Position', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Velocity comparison
axes[1].plot(time, velocity_est, 'r-', linewidth=2, alpha=0.7, label='Filtered')
axes[1].plot(time, velocity_smooth, 'g-', linewidth=2, label='Smoothed')
axes[1].plot(time, true_velocity, 'b--', linewidth=2, label='True')
axes[1].set_xlabel('Time', fontsize=12)
axes[1].set_ylabel('Velocity', fontsize=12)
axes[1].set_title('Filtering vs Smoothing: Velocity', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compute MSE
mse_filt = np.mean((position_est - true_position)**2)
mse_smooth = np.mean((position_smooth - true_position)**2)
print(f"\nPosition MSE - Filtered: {mse_filt:.3f}, Smoothed: {mse_smooth:.3f}")
print(f"Improvement: {(1 - mse_smooth/mse_filt)*100:.1f}%")

**Key Observation:** Smoothed estimates are more accurate (use future data)

## 6. Application: Commodity Price Filtering

Apply Kalman filter to real commodity data.

In [ ]:
import yfinance as yf

# Fetch crude oil data
oil = yf.download('CL=F', start='2023-01-01', end='2024-01-01', progress=False)
prices = oil['Adj Close'].dropna().values

print(f"Loaded {len(prices)} oil prices")
print(f"Price range: ${prices.min():.2f} - ${prices.max():.2f}")

In [ ]:
# Local level model: price follows random walk with measurement noise
T_oil = len(prices)

# Parameters (tune these!)
sigma_obs = 2.0   # Measurement noise (noisy observations)
sigma_state = 1.0  # State evolution (how much price level changes)

# Build Kalman filter
kf_oil = ScalarKalmanFilter(
    F=1.0,
    H=1.0,
    Q=sigma_state**2,
    R=sigma_obs**2,
    x0=prices[0],
    P0=10.0
)

# Filter prices
for p in prices:
    kf_oil.step(p)

filtered_prices = np.array(kf_oil.history['x_filt'])
print("Filtering complete")

In [ ]:
# Visualize filtered prices
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

time_oil = np.arange(T_oil)

# Prices
axes[0].plot(time_oil, prices, 'o-', alpha=0.5, markersize=3, label='Raw Prices')
axes[0].plot(time_oil, filtered_prices, 'r-', linewidth=2, label='Kalman Filtered')
axes[0].set_ylabel('Price ($)', fontsize=12)
axes[0].set_title('Crude Oil: Kalman Filtering', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Innovations
innovations_oil = np.array(kf_oil.history['innovation'])
axes[1].plot(time_oil, innovations_oil, 'o-', linewidth=1, markersize=4, alpha=0.7)
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('Trading Day', fontsize=12)
axes[1].set_ylabel('Innovation', fontsize=12)
axes[1].set_title('Innovations (should be uncorrelated)', fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Exercise 1: Tune Filter Parameters

**Task:** Find optimal $Q$ and $R$ for commodity price filtering.

1. Try different $(Q, R)$ combinations
2. Compute out-of-sample forecast error
3. Plot error vs parameters
4. Identify best configuration

**Hint:** Grid search over $\sigma_{state} \in [0.5, 1, 2, 5]$ and $\sigma_{obs} \in [0.5, 1, 2, 5]$

In [ ]:
# YOUR CODE HERE

# YOUR CODE: Grid search for optimal Q and R
best_params = None

---

## Exercise 2: Local Linear Trend Model

**Task:** Extend to local linear trend (level + slope).

**Model:**
$$\begin{align}
\mu_t &= \mu_{t-1} + \beta_{t-1} + \eta_t \\
\beta_t &= \beta_{t-1} + \zeta_t \\
y_t &= \mu_t + \epsilon_t
\end{align}$$

State: $\mathbf{x}_t = [\mu_t, \beta_t]^T$

1. Define matrices $F$, $H$, $Q$, $R$
2. Apply multivariate Kalman filter
3. Extract level and slope estimates
4. Compare with local level model

In [ ]:
# YOUR CODE HERE

# YOUR CODE: Implement local linear trend model

---

## Exercise 3: Missing Data Handling

**Task:** Modify Kalman filter to skip missing observations.

1. Create price series with 10% missing (set to NaN)
2. Implement skip logic (predict only, no update when missing)
3. Show that filter interpolates missing values
4. Compare MSE with complete data case

In [ ]:
# YOUR CODE HERE

# YOUR CODE: Handle missing data

---

## Summary

### Key Takeaways

1. **Kalman filter** is optimal linear estimator for Gaussian systems
2. **Two-step algorithm**: Predict (time update), Update (measurement update)
3. **Kalman gain** balances prior belief vs new data
4. **Filtering** = online (uses past), **Smoothing** = offline (uses all data)
5. **Innovations** diagnostics reveal model misspecification
6. **Multivariate** extension handles multiple states/observations

### When to Use Kalman Filter

- Tracking/denoising with known dynamics
- Real-time state estimation
- Systems with measurement noise
- Linear Gaussian state space models

### Limitations

- Assumes linearity (use EKF/UKF for nonlinear)
- Assumes Gaussian noise (use particle filter for non-Gaussian)
- Requires known parameters (use EM or Bayesian methods to learn)

### What's Next

Next notebook: **Stochastic Volatility Models** - Non-Gaussian state space with particle filtering

### Additional Resources

- Kalman (1960): Original paper
- Welch & Bishop: "An Introduction to the Kalman Filter" (excellent tutorial)
- Durbin & Koopman (2012): *Time Series Analysis by State Space Methods*

---